In [1]:
import pandas as pd
import numpy as np
from tqdm import tqdm
from DATA.stock_invest_function import *
from statsmodels.stats.diagnostic import acorr_ljungbox
from itertools import product
from statsmodels.tsa.statespace.sarimax import SARIMAX


import warnings
warnings.filterwarnings("ignore")

In [2]:
import matplotlib
import matplotlib.pyplot as plt
matplotlib.rc('font', family='Malgun Gothic')
plt.rcParams['axes.unicode_minus'] = False

In [3]:
# DB 접속 정보 설정
db_info = {
    'user': 'stox7412',         # 예: 'root'
    'password': 'Apt106503!~', # 예: '1234'
    'host': '192.168.0.230',
    # 'host': 'hystox74.synology.me',         # 예: 'localhost' 또는 IP
    'port': '3307',              # 기본 포트는 보통 3306
    'database': 'investar'        # 예: 'trade_data'
}

In [4]:
def forecast_future_4q_with_sarima(df, date_col='date', value_col='endog_var', use_log=True):
    # 날짜 결측값 체크
    if df[date_col].isna().any():
        print(f"❌ 날짜에 결측치 존재 → 예측 불가: {df[date_col].isna().sum()}개")
        return pd.DataFrame({
            'date': pd.date_range(start=pd.Timestamp.today(), periods=4, freq='Q'),
            'revenue_forecast': [np.nan] * 4
        })

    # 시계열 설정
    ts = df.set_index(date_col)[value_col].asfreq('Q')

    if ts.isna().any():
        print("❌ 값에 결측치가 존재 → 보간 또는 제거 필요")
        return pd.DataFrame({
            'date': pd.date_range(start=ts.index[-1] + pd.offsets.QuarterEnd(), periods=4, freq='Q'),
            'revenue_forecast': [np.nan] * 4
        })

    if use_log:
        ts_transformed = np.log(ts)
    else:
        ts_transformed = ts

    # SARIMA 파라미터 탐색
    p = d = q = P = D = Q = [0, 1]
    s = 4
    param_combinations = list(product(p, d, q))
    seasonal_combinations = list(product(P, D, Q))
    total_combinations = list(product(param_combinations, seasonal_combinations))

    best_aic = np.inf
    best_model = None

    for (order, seasonal) in total_combinations:
        seasonal_order = (*seasonal, s)
        try:
            model = SARIMAX(ts_transformed, order=order, seasonal_order=seasonal_order)
            result = model.fit(disp=False)
            if result.aic < best_aic:
                best_aic = result.aic
                best_model = result
        except:
            continue

    if best_model is None:
        print("❌ 적합한 모델을 찾지 못했습니다.")
        return pd.DataFrame({
            'date': pd.date_range(start=ts.index[-1] + pd.offsets.QuarterEnd(), periods=4, freq='Q'),
            'revenue_forecast': [np.nan] * 4
        })

    # 예측 수행
    forecast_log = best_model.forecast(steps=4)
    forecast = np.exp(forecast_log) if use_log else forecast_log

    result_df = pd.DataFrame({
        'date': forecast.index,
        'revenue_forecast': forecast.values
    })

    return result_df


def forecast_multiple_symbols(filtered_df, date_col='date', value_col='value'):
    """
    여러 종목(ticker)의 시계열 데이터를 SARIMA 모델로 4분기 예측한 결과를 결합하여 반환합니다.

    Parameters:
        filtered_df (pd.DataFrame): 'date', 'ticker', 'value' 컬럼 포함
        date_col (str): 날짜 컬럼명
        value_col (str): 예측 대상 값 컬럼명

    Returns:
        pd.DataFrame: 종목별 예측 결과 (date, ticker, value, forecast)
    """
    result_list = []

    for symbol in filtered_df['ticker'].unique():
        sub_df = filtered_df[filtered_df['ticker'] == symbol].copy()
        sub_df = sub_df[[date_col, value_col]].rename(columns={value_col: 'endog_var'})

        # forecast 함수 호출
        forecast_df = forecast_future_4q_with_sarima(sub_df, date_col=date_col)

        # 기본 컬럼 이름 확인 후 무조건 rename
        forecast_df = forecast_df.rename(columns={
            'tic_name': 'ticker',  # 반드시 변환
            'revenue_forecast': 'value'
        })

        # ticker가 없는 경우 대비하여 무조건 지정
        forecast_df['ticker'] = symbol

        # forecast 플래그 추가
        forecast_df['forecast'] = 1

        result_list.append(forecast_df)

    final_df = pd.concat(result_list, ignore_index=True)

    # 디버깅용 출력 (원할 경우 주석 처리)
    print("📌 최종 컬럼 목록:", final_df.columns.tolist())

    return final_df[['date', 'ticker', 'value', 'forecast']]


In [5]:
# tic_name = "A000660"

fs_df = fetch_table_data(db_info, "korea_fs_data")
fs_df.rename(columns={'Date': 'date'}, inplace=True)

# 1. indicator 필터링
target_indicator = '매출액(천원)'
filtered_df = fs_df[fs_df['indicator'] == target_indicator].copy()

# 2. 날짜 정제 및 정렬
filtered_df['date'] = pd.to_datetime(filtered_df['date'])
filtered_df.sort_values(by='date', inplace=True)
filtered_df.rename(columns={'symbol': 'ticker'}, inplace=True)

# 3. value 컬럼이 있는지 확인 및 타입 강제
if 'value' not in filtered_df.columns:
    raise KeyError("'value' 컬럼이 없습니다.")

filtered_df['value'] = pd.to_numeric(filtered_df['value'], errors='coerce')

# 4. 피벗 테이블 생성 (행: date, 열: Symbol, 값: value)
pivot_df = filtered_df.pivot_table(
    index='date',
    columns='ticker',
    values='value',
    aggfunc='first'  # 중복 방지
)

# 5. 전년 동분기 대비 변화율 계산 (4분기 전 대비)
fs_yoy_growth_df = pivot_df.pct_change(periods=4) * 100

# endog_df = pivot_df[[tic_name]].reset_index()

✅ 'korea_fs_data' 테이블에서 5584577건의 데이터를 가져왔습니다.


In [ ]:
filtered_df

In [6]:
def compute_yoy_and_ttm_growth(df):
    # 정렬
    df = df.sort_values(by=['ticker', 'date']).reset_index(drop=True)

    # YoY 성장률: 4분기 전 대비 비율
    df['yoy_growth'] = df.groupby('ticker')['value'].transform(lambda x: x.pct_change(periods=4))

    # TTM 값 계산: 최근 4개 분기의 평균
    df['ttm_current'] = df.groupby('ticker')['value'].transform(lambda x: x.rolling(window=4).mean())

    # 1년 전 TTM 평균
    df['ttm_past'] = df.groupby('ticker')['ttm_current'].shift(4)

    # TTM 성장률 계산
    df['TTM_growth'] = (df['ttm_current'] - df['ttm_past']) / df['ttm_past'].abs()

    # 불필요한 중간 컬럼 제거
    # df = df.drop(columns=['ttm_current', 'ttm_past'])

    return df


In [9]:
tic_list = filtered_df['ticker'].unique().tolist()

In [10]:
sorted_df = filtered_df[filtered_df['ticker'].isin(tic_list)].sort_values(by=['ticker','date'])
sorted_df_resize = sorted_df[['date', 'ticker', 'value']]
sorted_df_resize['forecast'] = 0
forecast_df = forecast_multiple_symbols(sorted_df_resize)

❌ 값에 결측치가 존재 → 보간 또는 제거 필요
❌ 값에 결측치가 존재 → 보간 또는 제거 필요
❌ 적합한 모델을 찾지 못했습니다.
❌ 값에 결측치가 존재 → 보간 또는 제거 필요
❌ 적합한 모델을 찾지 못했습니다.
❌ 적합한 모델을 찾지 못했습니다.
❌ 값에 결측치가 존재 → 보간 또는 제거 필요
❌ 값에 결측치가 존재 → 보간 또는 제거 필요
❌ 값에 결측치가 존재 → 보간 또는 제거 필요
❌ 적합한 모델을 찾지 못했습니다.
❌ 값에 결측치가 존재 → 보간 또는 제거 필요
❌ 적합한 모델을 찾지 못했습니다.
❌ 값에 결측치가 존재 → 보간 또는 제거 필요
❌ 값에 결측치가 존재 → 보간 또는 제거 필요
❌ 적합한 모델을 찾지 못했습니다.
❌ 값에 결측치가 존재 → 보간 또는 제거 필요
❌ 적합한 모델을 찾지 못했습니다.
❌ 적합한 모델을 찾지 못했습니다.
❌ 적합한 모델을 찾지 못했습니다.
❌ 적합한 모델을 찾지 못했습니다.
❌ 적합한 모델을 찾지 못했습니다.
❌ 적합한 모델을 찾지 못했습니다.
❌ 적합한 모델을 찾지 못했습니다.
❌ 값에 결측치가 존재 → 보간 또는 제거 필요
❌ 적합한 모델을 찾지 못했습니다.
❌ 값에 결측치가 존재 → 보간 또는 제거 필요
❌ 적합한 모델을 찾지 못했습니다.
❌ 적합한 모델을 찾지 못했습니다.
❌ 적합한 모델을 찾지 못했습니다.
❌ 값에 결측치가 존재 → 보간 또는 제거 필요
❌ 값에 결측치가 존재 → 보간 또는 제거 필요
❌ 적합한 모델을 찾지 못했습니다.
❌ 적합한 모델을 찾지 못했습니다.
❌ 적합한 모델을 찾지 못했습니다.
❌ 적합한 모델을 찾지 못했습니다.
❌ 적합한 모델을 찾지 못했습니다.
❌ 값에 결측치가 존재 → 보간 또는 제거 필요
❌ 적합한 모델을 찾지 못했습니다.
❌ 적합한 모델을 찾지 못했습니다.
❌ 적합한 모델을 찾지 못했습니다.
❌ 적합한 모델을 찾지 못했습니다.
❌ 적합한 모델을 찾지 못했습니다.
❌ 적합한 모델을 찾지 못했습니다.
❌ 적합한 모델을 찾지 못했습니다.
❌ 값에 결측치가 존재 → 

In [12]:
revenue_df_with_forecast = pd.concat([sorted_df_resize, forecast_df])
revenue_growth = result_df = compute_yoy_and_ttm_growth(revenue_df_with_forecast)

In [20]:
revenue_growth.to_csv('한군기업배출예측_단순SARIMA.csv', index=False)